# 09 - Model comparison, round 3: targeted features + XGBoost

Rerun the model comparison on the 39-feature set from `08_targeted_features.ipynb` (34 + king-attack pressure, pins, game phase), and add XGBoost alongside `HistGradientBoostingClassifier` — both are histogram-based gradient boosting, so expect a small difference at most, not a different tier of performance.

In [1]:
import sys
sys.path.append('..')

import time
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
import xgboost as xgb

from src.features import extract_features_v3

train_df = pd.read_csv('../data/processed/train_features_v3.csv', low_memory=False)
test_df = pd.read_csv('../data/processed/test_features_v3.csv', low_memory=False)

feature_cols = list(extract_features_v3(train_df['fen'].iloc[0]).keys())
X_train, y_train = train_df[feature_cols], train_df['white_win']
X_test, y_test = test_df[feature_cols], test_df['white_win']

print(f'{len(feature_cols)} features')
X_train.shape, X_test.shape

39 features


((822936, 39), (208378, 39))

In [2]:
def evaluate_model(name, model, X_tr, X_te):
    start = time.time()
    model.fit(X_tr, y_train)
    fit_seconds = time.time() - start

    preds = model.predict(X_te)
    metrics = {
        'model': name,
        'accuracy': accuracy_score(y_test, preds),
        'precision': precision_score(y_test, preds),
        'recall': recall_score(y_test, preds),
        'f1': f1_score(y_test, preds),
        'recall_black_win': recall_score(y_test, preds, pos_label=0),
        'fit_seconds': fit_seconds,
    }

    print(f'{name}  (fit in {fit_seconds:.1f}s)')
    print(classification_report(y_test, preds, target_names=['black_win', 'white_win']))

    return metrics, model


results = []

scaler = StandardScaler().fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

metrics, _ = evaluate_model('Logistic Regression', LogisticRegression(max_iter=1000), X_train_scaled, X_test_scaled)
results.append(metrics)

Logistic Regression  (fit in 0.7s)
              precision    recall  f1-score   support

   black_win       0.67      0.57      0.62    101264
   white_win       0.64      0.73      0.68    107114

    accuracy                           0.65    208378
   macro avg       0.66      0.65      0.65    208378
weighted avg       0.66      0.65      0.65    208378



In [3]:
rf = RandomForestClassifier(n_estimators=200, max_depth=12, n_jobs=-1, random_state=42)
metrics, rf = evaluate_model('Random Forest', rf, X_train, X_test)
results.append(metrics)

hgb = HistGradientBoostingClassifier(max_iter=300, random_state=42)
metrics, hgb = evaluate_model('Gradient Boosting (HistGB)', hgb, X_train, X_test)
results.append(metrics)

xgb_model = xgb.XGBClassifier(
    n_estimators=300, max_depth=6, learning_rate=0.1,
    eval_metric='logloss', n_jobs=-1, random_state=42,
)
metrics, xgb_model = evaluate_model('XGBoost', xgb_model, X_train, X_test)
results.append(metrics)

Random Forest  (fit in 17.8s)
              precision    recall  f1-score   support

   black_win       0.69      0.53      0.60    101264
   white_win       0.64      0.78      0.70    107114

    accuracy                           0.66    208378
   macro avg       0.66      0.65      0.65    208378
weighted avg       0.66      0.66      0.65    208378



Gradient Boosting (HistGB)  (fit in 9.9s)
              precision    recall  f1-score   support

   black_win       0.68      0.56      0.62    101264
   white_win       0.65      0.75      0.70    107114

    accuracy                           0.66    208378
   macro avg       0.66      0.66      0.66    208378
weighted avg       0.66      0.66      0.66    208378



XGBoost  (fit in 3.4s)
              precision    recall  f1-score   support

   black_win       0.68      0.57      0.62    101264
   white_win       0.65      0.75      0.69    107114

    accuracy                           0.66    208378
   macro avg       0.66      0.66      0.66    208378
weighted avg       0.66      0.66      0.66    208378



## Before/after: round 2 (34 features) vs. round 3 (39 features)

In [4]:
# round-2 numbers from 07_model_comparison_v2.ipynb, hardcoded for a direct comparison
round2 = pd.DataFrame([
    {'model': 'Logistic Regression', 'accuracy': 0.6536, 'f1': 0.6831, 'recall_black_win': 0.5768},
    {'model': 'Random Forest', 'accuracy': 0.6578, 'f1': 0.6995, 'recall_black_win': 0.5343},
    {'model': 'Gradient Boosting (HistGB)', 'accuracy': 0.6602, 'f1': 0.6951, 'recall_black_win': 0.5613},
]).set_index('model')

round3 = pd.DataFrame(results).set_index('model')[['accuracy', 'f1', 'recall_black_win']]

comparison = round2.join(round3, lsuffix='_v2_34feat', rsuffix='_v3_39feat', how='right')
comparison[[
    'accuracy_v2_34feat', 'accuracy_v3_39feat',
    'f1_v2_34feat', 'f1_v3_39feat',
    'recall_black_win_v2_34feat', 'recall_black_win_v3_39feat',
]].round(4)

,accuracy_v2_34feat,accuracy_v3_39feat,f1_v2_34feat,f1_v3_39feat,recall_black_win_v2_34feat,recall_black_win_v3_39feat
model,,,,,,
Logistic Regression,0.6536,0.6541,0.6831,0.6844,0.5768,0.5740
Random Forest,0.6578,0.6577,0.6995,0.6998,0.5343,0.5322
Gradient Boosting (HistGB),0.6602,0.6611,0.6951,0.6959,0.5613,0.5625
XGBoost,NaN,0.6593,NaN,0.6922,NaN,0.5681


## Where do the new features rank?
Random forest importances, full 39-feature set.

In [5]:
pd.DataFrame({
    'feature': feature_cols,
    'importance': rf.feature_importances_,
}).sort_values('importance', ascending=False).head(15)

,feature,importance
10,material_diff,0.335036
38,material_diff_x_endgame,0.106330
18,mobility_diff,0.105470
17,mobility_black,0.044413
27,passed_pawns_diff,0.039017
16,mobility_white,0.035275
28,king_safety_diff,0.031119
36,total_non_pawn_material,0.030016
34,attackers_near_king_diff,0.025679
0,white_pawns,0.021508


## Elo experiment: how much lift comes from player skill vs. board understanding?

`white_elo`/`black_elo` are player-level, not board-level — known before the game starts, and (unlike everything else so far) they describe *who's playing*, not *what the position looks like*. Adding them risks a model that's mostly learning "the higher-rated player usually wins" rather than reading the board. Report both configurations on the *same* rows so the comparison isolates the effect of adding `elo_diff`, not an artifact of a different row subset (some rows have `'?'` for Elo and get dropped either way).

In [6]:
for df in (train_df, test_df):
    df['white_elo_num'] = pd.to_numeric(df['white_elo'], errors='coerce')
    df['black_elo_num'] = pd.to_numeric(df['black_elo'], errors='coerce')
    df['elo_diff'] = df['white_elo_num'] - df['black_elo_num']

train_has_elo = train_df['elo_diff'].notna()
test_has_elo = test_df['elo_diff'].notna()

print(f'train rows with usable Elo: {train_has_elo.sum():,} / {len(train_df):,} ({train_has_elo.mean():.1%})')
print(f'test rows with usable Elo:  {test_has_elo.sum():,} / {len(test_df):,} ({test_has_elo.mean():.1%})')

train_elo_df = train_df[train_has_elo]
test_elo_df = test_df[test_has_elo]
y_train_elo = train_elo_df['white_win']
y_test_elo = test_elo_df['white_win']

train rows with usable Elo: 821,890 / 822,936 (99.9%)
test rows with usable Elo:  208,099 / 208,378 (99.9%)


In [7]:
def evaluate_with_labels(name, model, X_tr, X_te, y_tr, y_te):
    model.fit(X_tr, y_tr)
    preds = model.predict(X_te)
    metrics = {
        'model': name,
        'accuracy': accuracy_score(y_te, preds),
        'precision': precision_score(y_te, preds),
        'recall': recall_score(y_te, preds),
        'f1': f1_score(y_te, preds),
    }
    print(name)
    print(classification_report(y_te, preds, target_names=['black_win', 'white_win']))
    return metrics


elo_results = []

# without Elo: same 39 board features, restricted to the Elo-available rows
metrics = evaluate_with_labels(
    'Gradient Boosting, without Elo',
    HistGradientBoostingClassifier(max_iter=300, random_state=42),
    train_elo_df[feature_cols], test_elo_df[feature_cols], y_train_elo, y_test_elo,
)
elo_results.append(metrics)

# with Elo: same 39 board features + elo_diff
feature_cols_with_elo = feature_cols + ['elo_diff']
metrics = evaluate_with_labels(
    'Gradient Boosting, with Elo',
    HistGradientBoostingClassifier(max_iter=300, random_state=42),
    train_elo_df[feature_cols_with_elo], test_elo_df[feature_cols_with_elo], y_train_elo, y_test_elo,
)
elo_results.append(metrics)

pd.DataFrame(elo_results).set_index('model').round(4)

Gradient Boosting, without Elo
              precision    recall  f1-score   support

   black_win       0.68      0.57      0.62    101092
   white_win       0.65      0.75      0.70    107007

    accuracy                           0.66    208099
   macro avg       0.66      0.66      0.66    208099
weighted avg       0.66      0.66      0.66    208099



Gradient Boosting, with Elo
              precision    recall  f1-score   support

   black_win       0.72      0.70      0.71    101092
   white_win       0.73      0.74      0.74    107007

    accuracy                           0.73    208099
   macro avg       0.73      0.72      0.72    208099
weighted avg       0.73      0.73      0.73    208099



,accuracy,precision,recall,f1
model,,,,
"Gradient Boosting, without Elo",0.6614,0.6469,0.7518,0.6954
"Gradient Boosting, with Elo",0.7254,0.7275,0.7449,0.7361
